In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Systematic Review Analysis Template\n",
    "## Template for New Reviews\n",
    "\n",
    "This notebook provides a template for analyzing new systematic reviews."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Setup and Configuration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "sys.path.append('../src')\n",
    "\n",
    "from screening import SystematicReviewScreener\n",
    "from comparison import ScreeningComparator\n",
    "from performance import PerformanceCalculator\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import yaml\n",
    "import json\n",
    "from pathlib import Path\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Visualization setup\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "sns.set_palette(\"Set2\")\n",
    "\n",
    "# Display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)\n",
    "pd.set_option('display.max_colwidth', 150)\n",
    "\n",
    "print(\"✅ Libraries imported\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Configure Your Review"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configuration\n",
    "REVIEW_CONFIG = {\n",
    "    'name': 'YOUR_REVIEW_NAME',  # e.g., 'New Drug Review'\n",
    "    'data_dir': '../data/YOUR_REVIEW_DIR',  # e.g., '../data/new_review'\n",
    "    'config_file': '../config/YOUR_CONFIG.yaml',  # e.g., '../config/new_review.yaml'\n",
    "    'python_time': 0,  # Python screening time in minutes\n",
    "    'manual_time': 0   # Manual screening time in hours\n",
    "}\n",
    "\n",
    "# Update these paths\n",
    "RIS_FILE = f\"{REVIEW_CONFIG['data_dir']}/input.ris\"\n",
    "COVidence_INCLUDED = f\"{REVIEW_CONFIG['data_dir']}/covidence_included.csv\"\n",
    "COVidence_TOTAL = f\"{REVIEW_CONFIG['data_dir']}/covidence_total.csv\"\n",
    "\n",
    "print(\"🔧 Review Configuration:\")\n",
    "for key, value in REVIEW_CONFIG.items():\n",
    "    print(f\"  {key}: {value}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Data Loading and Exploration"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "try:\n",
    "    with open(REVIEW_CONFIG['config_file'], 'r') as f:\n",
    "        config = yaml.safe_load(f)\n",
    "    print(f\"✅ Configuration loaded: {config.get('review_name', 'Unknown')}\")\n",
    "except FileNotFoundError:\n",
    "    print(f\"❌ Config file not found: {REVIEW_CONFIG['config_file']}\")\n",
    "    print(\"   Using default configuration\")\n",
    "    config = {'review_name': REVIEW_CONFIG['name']}"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check data files\n",
    "data_files = {\n",
    "    'RIS File': RIS_FILE,\n",
    "    'Covidence Included': COVidence_INCLUDED,\n",
    "    'Covidence Total': COVidence_TOTAL\n",
    "}\n",
    "\n",
    "print(\"📁 Data File Status:\")\n",
    "for name, path in data_files.items():\n",
    "    if Path(path).exists():\n",
    "        size = Path(path).stat().st_size / 1024  # KB\n",
    "        print(f\"  ✓ {name}: {path} ({size:.1f} KB)\")\n",
    "    else:\n",
    "        print(f\"  ✗ {name}: NOT FOUND at {path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Screening Pipeline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize screener\n",
    "screener = SystematicReviewScreener(REVIEW_CONFIG['config_file'])\n",
    "\n",
    "# Run screening\n",
    "print(f\"\\n📚 Screening: {REVIEW_CONFIG['name']}\")\n",
    "try:\n",
    "    results = screener.screen(RIS_FILE, REVIEW_CONFIG['name'])\n",
    "    \n",
    "    print(f\"✅ Screening complete!\")\n",
    "    print(f\"   Total records: {len(screener.entries)}\")\n",
    "    print(f\"   Included: {len(results['include'])}\")\n",
    "    print(f\"   Maybe: {len(results['maybe'])}\")\n",
    "    print(f\"   Excluded: {len(results['exclude'])}\")\n",
    "    \n",
    "    # Save results\n",
    "    output_dir = Path(f\"../data/output/{REVIEW_CONFIG['name'].lower().replace(' ', '_')}\")\n",
    "    summary = screener.save_results(str(output_dir), REVIEW_CONFIG['name'])\n",
    "    \n",
    "    # Get CSV path for next steps\n",
    "    screening_csv = output_dir / f\"screening_decisions_{REVIEW_CONFIG['name'].lower().replace(' ', '_')}.csv\"\n",
    "    \n",
    "except Exception as e:\n",
    "    print(f\"❌ Screening failed: {e}\")\n",
    "    screening_csv = None"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Comparison with Covidence"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if screening_csv and screening_csv.exists():\n",
    "    print(f\"\\n🔍 Comparing with Covidence...\")\n",
    "    \n",
    "    comparator = ScreeningComparator(matching_strategy='accession_doi')\n",
    "    \n",
    "    try:\n",
    "        comparison_results = comparator.compare(str(screening_csv), COVidence_INCLUDED)\n",
    "        \n",
    "        print(f\"✅ Comparison complete!\")\n",
    "        print(f\"   Common studies: {comparison_results['common_studies']}\")\n",
    "        print(f\"   Python only: {comparison_results['python_only']}\")\n",
    "        print(f\"   Covidence only: {comparison_results['covidence_only']}\")\n",
    "        \n",
    "        if comparison_results['covidence_total'] > 0:\n",
    "            agreement = comparison_results['common_studies'] / comparison_results['covidence_total']\n",
    "            print(f\"   Agreement rate: {agreement:.1%}\")\n",
    "        \n",
    "        # Save comparison results\n",
    "        comparison_dir = output_dir / 'comparison'\n",
    "        comparison_dir.mkdir(exist_ok=True)\n",
    "        \n",
    "        with open(comparison_dir / 'comparison_results.json', 'w') as f:\n",
    "            json.dump(comparison_results, f, indent=2)\n",
    "            \n",
    "    except Exception as e:\n",
    "        print(f\"❌ Comparison failed: {e}\")\n",
    "else:\n",
    "    print(\"\\n⚠️  Skipping comparison - screening CSV not found\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Performance Calculation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if screening_csv and screening_csv.exists():\n",
    "    print(f\"\\n📊 Calculating performance...\")\n",
    "    \n",
    "    calculator = PerformanceCalculator()\n",
    "    \n",
    "    try:\n",
    "        metrics = calculator.calculate(str(screening_csv), COVidence_TOTAL)\n",
    "        \n",
    "        # Add time efficiency if provided\n",
    "        if REVIEW_CONFIG['python_time'] > 0 and REVIEW_CONFIG['manual_time'] > 0:\n",
    "            manual_minutes = REVIEW_CONFIG['manual_time'] * 60\n",
    "            time_saved = manual_minutes - REVIEW_CONFIG['python_time']\n",
    "            percent_saved = time_saved / manual_minutes if manual_minutes > 0 else 0\n",
    "            speed_increase = manual_minutes / REVIEW_CONFIG['python_time'] if REVIEW_CONFIG['python_time'] > 0 else 0\n",
    "            \n",
    "            metrics.update({\n",
    "                'python_time_minutes': REVIEW_CONFIG['python_time'],\n",
    "                'manual_time_hours': REVIEW_CONFIG['manual_time'],\n",
    "                'manual_time_minutes': manual_minutes,\n",
    "                'time_saved_minutes': time_saved,\n",
    "                'time_saved_hours': time_saved / 60,\n",
    "                'percent_time_saved': percent_saved,\n",
    "                'speed_increase': speed_increase\n",
    "            })\n",
    "        \n",
    "        print(f\"✅ Performance calculation complete!\")\n",
    "        print(f\"   Sensitivity: {metrics.get('sensitivity', 0):.1%}\")\n",
    "        print(f\"   Specificity: {metrics.get('specificity', 0):.1%}\")\n",
    "        print(f\"   Precision: {metrics.get('precision', 0):.1%}\")\n",
    "        print(f\"   Accuracy: {metrics.get('accuracy', 0):.1%}\")\n",
    "        \n",
    "        if 'speed_increase' in metrics:\n",
    "            print(f\"   Speed increase: {metrics['speed_increase']:.1f}x\")\n",
    "        \n",
    "        # Save performance results\n",
    "        performance_dir = output_dir / 'performance'\n",
    "        performance_dir.mkdir(exist_ok=True)\n",
    "        \n",
    "        with open(performance_dir / 'performance_metrics.json', 'w') as f:\n",
    "            json.dump(metrics, f, indent=2)\n",
    "        \n",
    "        # Generate visualization\n",
    "        calculator.generate_report(str(performance_dir), REVIEW_CONFIG['name'])\n",
    "        \n",
    "    except Exception as e:\n",
    "        print(f\"❌ Performance calculation failed: {e}\")\n",
    "else:\n",
    "    print(\"\\n⚠️  Skipping performance calculation - screening CSV not found\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Visualization"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Create simple visualizations\n",
    "print(f\"\\n🎨 Creating visualizations...\")\n",
    "\n",
    "fig, axes = plt.subplots(1, 3, figsize=(15, 5))\n",
    "\n",
    "# 1. Screening results\n",
    "if 'results' in locals():\n",
    "    categories = ['Included', 'Maybe', 'Excluded']\n",
    "    values = [len(results['include']), len(results['maybe']), len(results['exclude'])]\n",
    "    colors = ['#2ecc71', '#f39c12', '#e74c3c']\n",
    "    \n",
    "    axes[0].pie(values, labels=categories, colors=colors, autopct='%1.1f%%', startangle=90)\n",
    "    axes[0].set_title('Screening Decisions')\n",
    "\n",
    "# 2. Comparison results\n",
    "if 'comparison_results' in locals():\n",
    "    comparison_cats = ['Common', 'Python Only', 'Covidence Only']\n",
    "    comparison_vals = [\n",
    "        comparison_results['common_studies'],\n",
    "        comparison_results['python_only'],\n",
    "        comparison_results['covidence_only']\n",
    "    ]\n",
    "    \n",
    "    bars = axes[1].bar(comparison_cats, comparison_vals, color=['#3498db', '#2ecc71', '#e74c3c'])\n",
    "    axes[1].set_title('Comparison Results')\n",
    "    axes[1].set_ylabel('Number of Studies')\n",
    "    axes[1].tick_params(axis='x', rotation=45)\n",
    "    \n",
    "    for bar, val in zip(bars, comparison_vals):\n",
    "        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,\n",
    "                    str(val), ha='center')\n",
    "\n",
    "# 3. Performance metrics\n",
    "if 'metrics' in locals():\n",
    "    perf_metrics = ['Sensitivity', 'Specificity', 'Precision', 'Accuracy']\n",
    "    perf_values = [\n",
    "        metrics.get('sensitivity', 0),\n",
    "        metrics.get('specificity', 0),\n",
    "        metrics.get('precision', 0),\n",
    "        metrics.get('accuracy', 0)\n",
    "    ]\n",
    "    \n",
    "    bars = axes[2].bar(perf_metrics, perf_values, color='#9b59b6')\n",
    "    axes[2].set_ylim(0, 1)\n",
    "    axes[2].set_title('Performance Metrics')\n",
    "    axes[2].set_ylabel('Score')\n",
    "    axes[2].tick_params(axis='x', rotation=45)\n",
    "    \n",
    "    for bar, val in zip(bars, perf_values):\n",
    "        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,\n",
    "                    f'{val:.1%}', ha='center')\n",
    "\n",
    "plt.tight_layout()\n",
    "\n",
    "# Save figure\n",
    "if 'output_dir' in locals():\n",
    "    fig_path = output_dir / 'summary_visualization.png'\n",
    "    plt.savefig(fig_path, dpi=150, bbox_inches='tight')\n",
    "    print(f\"✅ Visualization saved: {fig_path}\")\n",
    "\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Summary Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Generate summary report\n",
    "print(f\"\\n📋 GENERATING SUMMARY REPORT\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "report = {\n",
    "    'review_name': REVIEW_CONFIG['name'],\n",
    "    'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),\n",
    "    'screening': {},\n",
    "    'comparison': {},\n",
    "    'performance': {},\n",
    "    'recommendations': []\n",
    "}\n",
    "\n",
    "# Add screening results\n",
    "if 'results' in locals():\n",
    "    report['screening'] = {\n",
    "        'total_records': len(screener.entries),\n",
    "        'included': len(results['include']),\n",
    "        'maybe': len(results['maybe']),\n",
    "        'excluded': len(results['exclude']),\n",
    "        'inclusion_rate': len(results['include']) / len(screener.entries) if len(screener.entries) > 0 else 0\n",
    "    }\n",
    "\n",
    "# Add comparison results\n",
    "if 'comparison_results' in locals():\n",
    "    report['comparison'] = {\n",
    "        'common_studies': comparison_results['common_studies'],\n",
    "        'python_only': comparison_results['python_only'],\n",
    "        'covidence_only': comparison_results['covidence_only'],\n",
    "        'agreement_rate': comparison_results['common_studies'] / comparison_results['covidence_total'] if comparison_results['covidence_total'] > 0 else 0\n",
    "    }\n",
    "\n",
    "# Add performance results\n",
    "if 'metrics' in locals():\n",
    "    report['performance'] = {\n",
    "        'sensitivity': metrics.get('sensitivity', 0),\n",
    "        'specificity': metrics.get('specificity', 0),\n",
    "        'precision': metrics.get('precision', 0),\n",
    "        'accuracy': metrics.get('accuracy', 0),\n",
    "        'f1_score': metrics.get('f1_score', 0)\n",
    "    }\n",
    "    \n",
    "    if 'speed_increase' in metrics:\n",
    "        report['performance']['efficiency'] = {\n",
    "            'speed_increase': metrics['speed_increase'],\n",
    "            'time_saved_hours': metrics.get('time_saved_hours', 0),\n",
    "            'percent_time_saved': metrics.get('percent_time_saved', 0)\n",
    "        }\n",
    "\n",
    "# Generate recommendations\n",
    "if 'comparison_results' in locals() and comparison_results['covidence_only'] > 10:\n",
    "    report['recommendations'].append(\n",
    "        f\"Review {comparison_results['covidence_only']} Covidence-only studies to improve sensitivity\"\n",
    "    )\n",
    "    \n",
    "if 'comparison_results' in locals() and comparison_results['python_only'] > 10:\n",
    "    report['recommendations'].append(\n",
    "        f\"Review {comparison_results['python_only']} Python-only studies to improve specificity\"\n",
    "    )\n",
    "\n",
    "if 'metrics' in locals() and metrics.get('sensitivity', 0) < 0.8:\n",
    "    report['recommendations'].append(\"Consider relaxing inclusion criteria to improve sensitivity\")\n",
    "\n",
    "if 'metrics' in locals() and metrics.get('specificity', 0) < 0.8:\n",
    "    report['recommendations'].append(\"Consider adding more specific exclusion criteria to improve specificity\")\n",
    "\n",
    "# Save report\n",
    "if 'output_dir' in locals():\n",
    "    report_file = output_dir / 'analysis_report.json'\n",
    "    with open(report_file, 'w') as f:\n",
    "        json.dump(report, f, indent=2)\n",
    "    \n",
    "    print(f\"✅ Summary report saved: {report_file}\")\n",
    "    print(f\"\\n📊 REPORT SUMMARY:\")\n",
    "    print(f\"   Review: {REVIEW_CONFIG['name']}\")\n",
    "    \n",
    "    if report['screening']:\n",
    "        print(f\"   Screening: {report['screening']['total_records']} records, \"\n",
    "              f\"{report['screening']['inclusion_rate']:.1%} inclusion rate\")\n",
    "    \n",
    "    if report['comparison']:\n",
    "        print(f\"   Comparison: {report['comparison']['agreement_rate']:.1%} agreement with Covidence\")\n",
    "    \n",
    "    if report['performance']:\n",
    "        print(f\"   Performance: Sensitivity={report['performance']['sensitivity']:.1%}, \"\n",
    "              f\"Specificity={report['performance']['specificity']:.1%}\")\n",
    "    \n",
    "    if report['recommendations']:\n",
    "        print(f\"\\n💡 Recommendations:\")\n",
    "        for rec in report['recommendations']:\n",
    "            print(f\"   • {rec}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Next Steps\n",
    "\n",
    "1. **Review the output files** in the generated directory\n",
    "2. **Check the screening decisions** for accuracy\n",
    "3. **Review discrepancies** between Python and Covidence\n",
    "4. **Adjust configuration** if needed and re-run\n",
    "5. **Use the results** for your systematic review manuscript"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "cochrane-screening",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}